In [98]:
from pathlib import Path

from scipy.cluster.hierarchy import fcluster

BASE_DIR = Path('../dataset/test-dataset')

In [99]:
from dataclasses import dataclass, field
import torch


@dataclass
class Image:
    filename: str
    camera_id: str

    path: Path = field(init=False, repr=False)
    tensor: torch.Tensor | None = field(init=False, default=None)

    def __post_init__(self):
        self.path = BASE_DIR / self.camera_id / self.filename


In [100]:
import os

def get_images():
    images: list[Image] = []

    for dir in os.listdir(BASE_DIR):
        if dir == '.DS_Store':
            continue
        images.extend([Image(filename=f, camera_id=dir) for f in os.listdir(BASE_DIR / dir)])
    
    return images

In [101]:
from torchvision import transforms
from PIL import Image as PILImage

TARGET_SIZE = 224

transform_pipeline = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def transform_image(image: Image):
    img = PILImage.open(image.path)
    image.tensor = transform_pipeline(img)

In [102]:
from torch.utils.data import Dataset

class ImageDataset(Dataset):
    def __init__(self, images: list[Image]) -> None:
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        transform_image(image)

        return str(image.path), image.tensor

In [103]:
from torch.utils.data import DataLoader

dataset = ImageDataset(images=get_images())

dataloader = DataLoader(
    dataset=dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [104]:
from transformers import AutoModel

local_model_dir = "../weights/dinov2_small"
embedding_model_id = "facebook/dinov2-small"

def load_embedding_model():
    model = AutoModel.from_pretrained(embedding_model_id)
    model.save_pretrained(local_model_dir)
    return model


def get_embedding_model_eval():
    if not os.path.exists(local_model_dir):
        return load_embedding_model()
    model = AutoModel.from_pretrained(local_model_dir, local_files_only=True)
    return model

## Извлечение признаков

In [105]:
from tqdm import tqdm
import numpy as np

embeddings_path = Path('../dataset/embeddings')

def extract_embeddings():
    if os.path.exists(embeddings_path / "image_embeddings_dinov2_small.npy"):
        return np.load(embeddings_path / "image_embeddings_dinov2_small.npy")

    device = torch.device('mps')

    all_embeddings = []

    model = get_embedding_model_eval()
    model.to(device)
    model.eval()

    for path, tensors in tqdm(dataloader, desc="Извлечение признаков"):
        tensors = tensors.to(device)
        with torch.no_grad():
            outputs = model(pixel_values=tensors)

        tokens = outputs.last_hidden_state[:, 0, :]
        embeddings = tokens.cpu().numpy()

        all_embeddings.append(embeddings)

    final_embeddings = np.vstack(all_embeddings)
    np.save(embeddings_path / "image_embeddings_dinov2_small.npy", final_embeddings)
    return final_embeddings

In [106]:
embeddings = extract_embeddings()
print(f"Форма {embeddings.shape}")

Форма (72, 384)


## Понижение размерности

In [141]:
from sklearn.decomposition import PCA

def dimensionality_reduction(n_components: int = 8):
    pca = PCA(n_components=n_components, random_state=42)
    reduced_embeddings = pca.fit_transform(embeddings)

    print(f"Форма {reduced_embeddings.shape}")
    print(f"Сохранение информации: {pca.explained_variance_ratio_.sum() * 100}")

    np.save(embeddings_path / f"image_embeddings_dinov2_small_{n_components}.npy", reduced_embeddings)

    return reduced_embeddings

In [180]:
reduced_embeddings = dimensionality_reduction(32)

Форма (72, 32)
Сохранение информации: 97.8763198852539


## Клатеризация по зонам

In [187]:
from sklearn.cluster import AgglomerativeClustering
import hdbscan

def agglomerative_clustering(n_clusters: int = 2):
    cluster_model = AgglomerativeClustering(linkage='ward', n_clusters=n_clusters)
    labels = cluster_model.fit_predict(reduced_embeddings)
    return labels

def hdbscan_clustering():
    cluster_model = hdbscan.HDBSCAN(
        min_cluster_size=4,
        min_samples=10,
        metric='euclidean',
        cluster_selection_epsilon=0.5
    )
    labels = cluster_model.fit_predict(reduced_embeddings)
    return labels

In [188]:
hdbscan_labels = hdbscan_clustering()
agglomerative_labels = agglomerative_clustering(n_clusters=len(set(hdbscan_labels)))

In [189]:
print(agglomerative_labels)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 4 4 4 4]
